# CV12 — Phân đoạn cả phần bị che

Input RGB và visible mask, target full mask. Mô hình chỉ học hidden region; công thức union đảm bảo xác suất full mask chứa visible mask nhị phân.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Input RGB và visible mask, target full mask. Mô hình chỉ học hidden region; công thức union đảm bảo xác suất full mask chứa visible mask nhị phân. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv12_amodal import Amodal, loss_fn, predict
images = torch.rand(2, 3, 32, 32, device=device)
full = torch.zeros(2, 1, 32, 32, device=device)
full[..., 8:24, 8:24] = 1
visible = full.clone()
visible[..., 16:, :] = 0
model = Amodal(d=16).to(device)
batch = (images, visible, full)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
with torch.no_grad():
    hidden_logits = model(images, visible)
    full_probability = predict(model, images, visible)
print('Hidden logits:', hidden_logits.shape)
print('Visible area / full target area:', visible.sum().item(), full.sum().item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Visible mask → hidden-region CNN → shape prior/copy-paste occlusion. Nếu đề không cấp visible mask thì cần detector/segmenter phía trước.


In [ ]:
model.eval()
with torch.no_grad():
    full_probability = predict(model, images, visible)
print('Contains the visible region:', (full_probability >= visible).all().item())
print('Baseline hidden-region recall is zero when predicting visible only.')
print('Evaluate full IoU and invisible-region IoU separately after training.')


## Thử nghiệm tiếp theo

Visible mask → hidden-region CNN → shape prior/copy-paste occlusion. Nếu đề không cấp visible mask thì cần detector/segmenter phía trước.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv12_amodal.py) và [tài liệu gốc](../REFERENCE.md).
